In [3]:
import joblib
import pandas as pd

model_path = "../models/failure_prediction_model.pkl"

failure_model = joblib.load(model_path)

print("Failure prediction model loaded successfully!")
print(type(failure_model))

Failure prediction model loaded successfully!
<class 'sklearn.pipeline.Pipeline'>


In [4]:
selected_features = [
    "Cycle",
    "Temperature_C",
    "Vibration_mm_s",
    "Pressure_bar",
    "Rotational_Speed_RPM",
    "Load_Fraction",
    "Torque_Nm",
    "Current_A",
    "Acoustic_dB",
    "Wear_Level",
    "Machine_Type"
]

print("Expected features:")
for feature in selected_features:
    print("-", feature)

Expected features:
- Cycle
- Temperature_C
- Vibration_mm_s
- Pressure_bar
- Rotational_Speed_RPM
- Load_Fraction
- Torque_Nm
- Current_A
- Acoustic_dB
- Wear_Level
- Machine_Type


In [12]:
df = pd.read_csv("../data/all_data/all_machines_combined.csv")

In [15]:
X = df[selected_features].copy()

print("Inference data shape:", X.shape)

display(X.head())

Inference data shape: (1930, 11)


,Cycle,Temperature_C,Vibration_mm_s,Pressure_bar,Rotational_Speed_RPM,Load_Fraction,Torque_Nm,Current_A,Acoustic_dB,Wear_Level,Machine_Type
0,1,68.987,0.2351,2.117,1518.64,0.6513,46.734,9.213,56.022,0.0000,CNC_Mill
1,2,69.431,0.1894,2.126,1517.67,0.5961,46.603,9.480,55.789,0.0000,CNC_Mill
2,3,68.779,0.2385,2.119,1520.83,0.6336,46.313,9.612,54.954,0.0022,CNC_Mill
3,4,68.926,0.2047,2.110,1531.69,0.6898,48.403,9.907,54.567,0.0000,CNC_Mill
4,5,68.704,0.2086,2.079,1516.70,0.5900,48.188,9.080,54.424,0.0006,CNC_Mill


In [16]:
sample_machine = X.iloc[[0]].copy()

display(sample_machine)

,Cycle,Temperature_C,Vibration_mm_s,Pressure_bar,Rotational_Speed_RPM,Load_Fraction,Torque_Nm,Current_A,Acoustic_dB,Wear_Level,Machine_Type
0,1,68.987,0.2351,2.117,1518.64,0.6513,46.734,9.213,56.022,0.0,CNC_Mill


In [17]:
prediction = failure_model.predict(sample_machine)[0]

failure_probability = failure_model.predict_proba(
    sample_machine
)[0][1]

print("Predicted Failure:", prediction)
print("Failure Probability:", failure_probability)

Predicted Failure: 0
Failure Probability: 0.00011016676


In [18]:
def predict_failure(machine_data):

    prediction = failure_model.predict(machine_data)[0]

    probability = failure_model.predict_proba(
        machine_data
    )[0][1]

    if probability >= 0.70:
        risk_level = "HIGH"
    elif probability >= 0.40:
        risk_level = "MEDIUM"
    else:
        risk_level = "LOW"

    return {
        "failure_prediction": int(prediction),
        "failure_probability": float(probability),
        "risk_level": risk_level
    }

In [19]:
result = predict_failure(sample_machine)

print(result)

{'failure_prediction': 0, 'failure_probability': 0.00011016675853170455, 'risk_level': 'LOW'}


In [20]:
test_samples = X.iloc[:10].copy()

predictions = failure_model.predict(test_samples)

probabilities = failure_model.predict_proba(
    test_samples
)[:, 1]

prediction_results = test_samples.copy()

prediction_results["Predicted_Failure"] = predictions
prediction_results["Failure_Probability"] = probabilities

prediction_results["Risk_Level"] = pd.cut(
    probabilities,
    bins=[-float("inf"), 0.40, 0.70, float("inf")],
    labels=["LOW", "MEDIUM", "HIGH"]
)

display(
    prediction_results[
        [
            "Machine_Type",
            "Cycle",
            "Wear_Level",
            "Predicted_Failure",
            "Failure_Probability",
            "Risk_Level"
        ]
    ]
)

,Machine_Type,Cycle,Wear_Level,Predicted_Failure,Failure_Probability,Risk_Level
0,CNC_Mill,1,0.0000,0,0.00011,LOW
1,CNC_Mill,2,0.0000,0,0.00011,LOW
2,CNC_Mill,3,0.0022,0,0.00011,LOW
3,CNC_Mill,4,0.0000,0,0.00011,LOW
4,CNC_Mill,5,0.0006,0,0.00011,LOW
5,CNC_Mill,6,0.0000,0,0.00011,LOW
6,CNC_Mill,7,0.0149,0,0.00011,LOW
7,CNC_Mill,8,0.0138,0,0.00011,LOW
8,CNC_Mill,9,0.0037,0,0.00011,LOW
9,CNC_Mill,10,0.0202,0,0.00011,LOW


In [21]:
prediction_results["Actual_Failure"] = df["Failure"].iloc[:10].values

In [22]:
display(
    prediction_results[
        [
            "Machine_Type",
            "Cycle",
            "Wear_Level",
            "Actual_Failure",
            "Predicted_Failure",
            "Failure_Probability",
            "Risk_Level"
        ]
    ]
)

,Machine_Type,Cycle,Wear_Level,Actual_Failure,Predicted_Failure,Failure_Probability,Risk_Level
0,CNC_Mill,1,0.0000,0,0,0.00011,LOW
1,CNC_Mill,2,0.0000,0,0,0.00011,LOW
2,CNC_Mill,3,0.0022,0,0,0.00011,LOW
3,CNC_Mill,4,0.0000,0,0,0.00011,LOW
4,CNC_Mill,5,0.0006,0,0,0.00011,LOW
5,CNC_Mill,6,0.0000,0,0,0.00011,LOW
6,CNC_Mill,7,0.0149,0,0,0.00011,LOW
7,CNC_Mill,8,0.0138,0,0,0.00011,LOW
8,CNC_Mill,9,0.0037,0,0,0.00011,LOW
9,CNC_Mill,10,0.0202,0,0,0.00011,LOW


In [23]:
all_predictions = failure_model.predict(X)

all_probabilities = failure_model.predict_proba(X)[:, 1]

print("Unique predicted classes:")
print(pd.Series(all_predictions).value_counts())

print("\nProbability statistics:")
print(pd.Series(all_probabilities).describe())

print("\nNumber of unique probability values:")
print(len(pd.Series(all_probabilities).unique()))

Unique predicted classes:
0    1809
1     121
Name: count, dtype: int64

Probability statistics:
count    1930.000000
mean        0.062960
std         0.236827
min         0.000060
25%         0.000110
50%         0.000110
75%         0.000185
max         0.998331
dtype: float64

Number of unique probability values:
597


In [24]:
risk_analysis = X.copy()

risk_analysis["Actual_Failure"] = df["Failure"].values
risk_analysis["Predicted_Failure"] = all_predictions
risk_analysis["Failure_Probability"] = all_probabilities

risk_analysis = risk_analysis.sort_values(
    "Failure_Probability",
    ascending=False
)

display(
    risk_analysis[
        [
            "Machine_Type",
            "Cycle",
            "Wear_Level",
            "Actual_Failure",
            "Predicted_Failure",
            "Failure_Probability"
        ]
    ].head(20)
)

,Machine_Type,Cycle,Wear_Level,Actual_Failure,Predicted_Failure,Failure_Probability
568,Hydraulic_Pump,309,0.9852,1,1,0.998331
1094,Electric_Motor,235,0.9734,1,1,0.998098
1695,Hydraulic_Press,266,0.9518,1,1,0.998060
1097,Electric_Motor,238,1.0000,1,1,0.998008
1096,Electric_Motor,237,0.9881,1,1,0.998008
1095,Electric_Motor,236,0.9874,1,1,0.998008
1098,Electric_Motor,239,0.9841,1,1,0.998008
1093,Electric_Motor,234,0.9755,1,1,0.998008
1704,Hydraulic_Press,275,1.0000,1,1,0.997895
1920,Industrial_Robot,216,0.9418,1,1,0.997871
